In [4]:
import numpy as np
import pandas as pd
import re
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict

# --- Configuration ---
TOP_K_RETRIEVAL = 10

# --- Tokenization ---
_DEVANAGARI_PATTERN = re.compile(r'[\u0900-\u0963\u0966-\u097F]+')

with open("/kaggle/input/nepal-bhasa-corpus/data/stopwords.txt", "r", encoding="utf-8") as f:
    _STOPWORDS = set(line.strip() for line in f)

def simple_tokenizer(text):
    """Tokenizes text handling Devanagari, English, and transliteration."""
    # Extract Devanagari words
    devanagari_words = _DEVANAGARI_PATTERN.findall(text)
    
    # Extract English/transliterated words (alphanumeric sequences)
    latin_words = re.findall(r'[a-zA-Z0-9]+', text)
    
    # Combine and filter stopwords
    all_words = devanagari_words + latin_words
    return [word.lower() for word in all_words if word.lower() not in _STOPWORDS]

def tokenizer_wrapper(text):
    """Wrapper for TfidfVectorizer compatibility."""
    return ' '.join(simple_tokenizer(text))

# --- Metric Calculation Functions ---
def calculate_ap(retrieved_indices: np.ndarray, gt_index: int, k: int) -> float:
    """Calculates Average Precision."""
    retrieved_indices_k = retrieved_indices[:k]
    
    if gt_index not in retrieved_indices_k:
        return 0.0
    
    rank = np.where(retrieved_indices_k == gt_index)[0][0] + 1
    return 1.0 / rank

def calculate_ndcg(retrieved_indices: np.ndarray, gt_index: int, k: int) -> float:
    """Calculates Normalized Discounted Cumulative Gain."""
    retrieved_indices_k = retrieved_indices[:k]
    relevance_scores = np.array([1.0 if idx == gt_index else 0.0 for idx in retrieved_indices_k])
    
    if len(relevance_scores) == 0:
        return 0.0
    
    discounts = np.log2(np.arange(2, len(relevance_scores) + 2))
    dcg = relevance_scores[0] + np.sum(relevance_scores[1:] / discounts[:-1])
    
    if gt_index in retrieved_indices_k:
        ideal_scores = np.zeros_like(relevance_scores)
        ideal_scores[0] = 1.0
        idcg = ideal_scores[0] + np.sum(ideal_scores[1:] / discounts[:-1])
    else:
        idcg = 1.0
    
    return dcg / idcg if idcg > 0 else 0.0

# --- TF-IDF Retrieval ---
def build_tfidf_index(corpus_texts: List[str]) -> tuple:
    """Builds TF-IDF index for the corpus."""
    print(f"Building TF-IDF index for {len(corpus_texts)} documents...")
    
    # Preprocess corpus
    processed_corpus = [tokenizer_wrapper(text) for text in corpus_texts]
    
    # Create TF-IDF vectorizer
    vectorizer = TfidfVectorizer(
        tokenizer=lambda x: x.split(),  # Already preprocessed
        lowercase=False,  # Already lowercased
        min_df=2,  # Ignore terms appearing in fewer than 2 documents
        max_df=0.95,  # Ignore terms appearing in more than 95% of documents
        sublinear_tf=True  # Use sublinear term frequency scaling
    )
    
    # Fit and transform corpus
    tfidf_matrix = vectorizer.fit_transform(processed_corpus)
    
    print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
    print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
    
    return vectorizer, tfidf_matrix

def retrieve_documents(query: str, vectorizer: TfidfVectorizer, tfidf_matrix, top_k: int = TOP_K_RETRIEVAL) -> tuple:
    """Retrieves top-k documents for a query using TF-IDF."""
    # Preprocess query
    processed_query = tokenizer_wrapper(query)
    
    # Transform query to TF-IDF vector
    query_vector = vectorizer.transform([processed_query])
    
    # Calculate cosine similarity
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    # Get top-k indices (sorted by similarity, descending)
    top_indices = np.argsort(similarities)[::-1][:top_k]
    top_scores = similarities[top_indices]
    
    return top_indices, top_scores

# --- Evaluation ---
def evaluate_retrieval(test_data: pd.DataFrame, vectorizer: TfidfVectorizer, 
                      tfidf_matrix, corpus_id_map: Dict[str, int]) -> Dict[str, float]:
    """Evaluates TF-IDF retrieval performance."""
    
    mrr_total = 0.0
    map_total = 0.0
    ndcg_total = 0.0
    recall_at_k_hits = 0
    top_1_hits = 0
    top_5_hits = 0
    
    num_queries = len(test_data)
    
    for _, row in test_data.iterrows():
        query = row['Query']
        gt_doc_id = row['relevant docs']
        
        # Retrieve documents
        retrieved_indices, scores = retrieve_documents(query, vectorizer, tfidf_matrix, TOP_K_RETRIEVAL)
        
        # Get ground truth index
        gt_index = corpus_id_map.get(gt_doc_id)
        
        if gt_index is None:
            continue
        
        # Check if relevant document is retrieved
        if gt_index in retrieved_indices:
            rank = np.where(retrieved_indices == gt_index)[0][0] + 1
            
            mrr_total += 1.0 / rank
            recall_at_k_hits += 1
            map_total += calculate_ap(retrieved_indices, gt_index, TOP_K_RETRIEVAL)
            ndcg_total += calculate_ndcg(retrieved_indices, gt_index, TOP_K_RETRIEVAL)
            
            if rank == 1:
                top_1_hits += 1
            if rank <= 5:
                top_5_hits += 1
    
    results = {
        "MRR": mrr_total / num_queries,
        "MAP": map_total / num_queries,
        f"NDCG@{TOP_K_RETRIEVAL}": ndcg_total / num_queries,
        f"Recall@{TOP_K_RETRIEVAL}": recall_at_k_hits / num_queries,
        "Accuracy@1": top_1_hits / num_queries,
        "Accuracy@5": top_5_hits / num_queries,
    }
    
    return results

# --- Main Execution ---
def run_tfidf_retrieval():
    # Load data
    corpus_df = pd.read_csv('/kaggle/input/nepal-bhasa-corpus/data_output/corpus_data.csv')
    test_relevance = pd.read_csv('/kaggle/input/nepal-bhasa-corpus/data_output/test_relevance.csv')
    
    # Create mappings
    corpus_texts = corpus_df['text'].tolist()
    doc_ids = corpus_df['DocID'].tolist()
    doc_id_to_index = {doc_id: i for i, doc_id in enumerate(doc_ids)}
    
    # Build TF-IDF index
    vectorizer, tfidf_matrix = build_tfidf_index(corpus_texts)
    
    # Evaluate
    print("--- Running Evaluation on Test Set ---")
    results = evaluate_retrieval(test_relevance, vectorizer, tfidf_matrix, doc_id_to_index)
    
    print("\n✅ TF-IDF Retrieval Performance:")
    print("---------------------------------")
    for metric, value in results.items():
        print(f"{metric:<15}: {value:.4f}")
    print("---------------------------------")

if __name__ == '__main__':
    run_tfidf_retrieval()

Building TF-IDF index for 80380 documents...


/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:528: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


TF-IDF matrix shape: (80380, 95537)
Vocabulary size: 95537
--- Running Evaluation on Test Set ---

✅ TF-IDF Retrieval Performance:
---------------------------------
MRR            : 0.5678
MAP            : 0.5678
NDCG@10        : 0.6299
Recall@10      : 0.6767
Accuracy@1     : 0.5067
Accuracy@5     : 0.6533
---------------------------------
